In [2]:
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import datetime
from itertools import chain
import os
import tqdm
import glob

In [5]:
os.getcwd()

'/home/mds_backfill'

In [6]:
os.chdir('/home/mds_backfill/rpc_feature_extraction/rpc_model/sample_data_individual_feat')

In [7]:
files = glob.glob('*.{}'.format('csv'))
files.sort()
files

['bounce_item_w1_2023-01-09.csv',
 'bounce_item_w1_2023-01-16.csv',
 'bounce_item_w2_2023-01-09.csv',
 'bounce_item_w2_2023-01-16.csv',
 'catalog_w1_2023-01-09.csv',
 'catalog_w1_2023-01-16.csv',
 'df_header_2023-01-02.csv',
 'df_header_2023-01-09.csv',
 'df_header_2023-01-16.csv',
 'semcoop_l_2023-01-02.csv',
 'semcoop_l_2023-01-09.csv',
 'semcoop_l_2023-01-16.csv',
 'semcoop_v12_2023-01-02.csv',
 'semcoop_v12_2023-01-09.csv',
 'semcoop_v12_2023-01-16.csv',
 'semcoop_v18_2023-01-02.csv',
 'semcoop_v18_2023-01-09.csv',
 'semcoop_v18_2023-01-16.csv',
 'semcoop_v2_2023-01-02.csv',
 'semcoop_v2_2023-01-09.csv',
 'semcoop_v2_2023-01-16.csv',
 'semcoop_v3_2023-01-02.csv',
 'semcoop_v3_2023-01-09.csv',
 'semcoop_v3_2023-01-16.csv',
 'semcoop_v4_2023-01-02.csv',
 'semcoop_v4_2023-01-09.csv',
 'semcoop_v4_2023-01-16.csv',
 'semcoop_v5_2023-01-02.csv',
 'semcoop_v5_2023-01-09.csv',
 'semcoop_v5_2023-01-16.csv',
 'semcoop_v9_2023-01-02.csv',
 'semcoop_v9_2023-01-09.csv',
 'semcoop_v9_2023-01-16.

In [8]:
import pandas as pd
start_date = '2022-12-19'
end_date = '2023-01-16'
ref_dates = pd.date_range(start=pd.to_datetime(start_date), end=pd.to_datetime(end_date), freq='7D')
ref_dates = [x.strftime('%Y-%m-%d') for x in ref_dates]

In [11]:
ref_dates[-2:][::-1]

['2023-01-16', '2023-01-09']

In [ ]:
for date in ref_dates[-2:][::-1]:
    print('working on date '+date)
    # read header
    file = 'df_header_'+date+'.csv'
    df_feat_all = pd.read_csv(file, index_col=0)
    
    df_feat_all = df_feat_all[(~pd.isnull(df_feat_all['catalog_item_id']))&(~pd.isnull(df_feat_all['seller_id']))]
    df_feat_all['catalog_item_id'] = df_feat_all['catalog_item_id'].astype('int')
    df_feat_all['catalog_item_id'] = df_feat_all['catalog_item_id'].astype('str')
    df_feat_all['seller_id'] = df_feat_all['seller_id'].astype('int')
    df_feat_all['seller_id'] = df_feat_all['seller_id'].astype('str')
    df_feat_all.drop_duplicates(subset = 'adid', inplace=True)
    
    # drop off duplicates catalog_item_id, seller_id
    
#     # read targets
#     file = 'sem_l_'+date+'.csv'
#     df_tmp = pd.read_csv(file, index_col=0)
#     df_feat_all = df_tmp.merge(df_feat_all, left_on = 'catalog_item_id', right_on = 'catalog_item_id', how='left')
    
    # read sem coop features
    files_to_read = [x for x in files if x[-14:-4] == date and x[:4]=='semc']
    
    # left join to ensure item coverage is based on target coverage
    for file in files_to_read:
        print('merge sem coop features')
        df_tmp = pd.read_csv(file, index_col=0)
        df_tmp = df_tmp[(~pd.isnull(df_tmp['catalog_item_id']))&(~pd.isnull(df_tmp['seller_id']))]
        df_tmp['catalog_item_id'] = df_tmp['catalog_item_id'].astype('int')
        df_tmp['catalog_item_id'] = df_tmp['catalog_item_id'].astype('str')
        df_tmp['seller_id'] = df_tmp['seller_id'].astype('int')
        df_tmp['seller_id'] = df_tmp['seller_id'].astype('str')
        df_tmp.drop_duplicates(subset = 'adid', inplace=True)
        df_feat_all = df_feat_all.merge(df_tmp, on=['catalog_item_id', 'seller_id', 'source_id', 'adid', 'is_mobile'], how='left')
        del df_tmp
        
    # read catalog feature
    files_to_read = [x for x in files if x[-14:-4] == date and x[:7]=='catalog']
    for file in files_to_read:
        print('merge catalog features')
        df_tmp = pd.read_csv(file, index_col=0)
        df_tmp = df_tmp[(~pd.isnull(df_tmp['catalog_item_id']))&(~pd.isnull(df_tmp['seller_id']))]
        df_tmp['catalog_item_id'] = df_tmp['catalog_item_id'].astype('int')
        df_tmp['catalog_item_id'] = df_tmp['catalog_item_id'].astype('str')
        df_tmp['seller_id'] = df_tmp['seller_id'].astype('int')
        df_tmp['seller_id'] = df_tmp['seller_id'].astype('str')
        df_tmp.drop_duplicates(subset = ['catalog_item_id','seller_id'], inplace=True)
        df_feat_all = df_feat_all.merge(df_tmp, on=['catalog_item_id', 'seller_id'], how='left')
        del df_tmp
        
    # read site features
    files_to_read = [x for x in files if x[-14:-4] == date and x[:4]=='site' and 'item' in x]
    for file in files_to_read:
        print('merge site features')
        df_tmp = pd.read_csv(file, index_col=0)
        df_tmp = df_tmp[(~pd.isnull(df_tmp['catalog_item_id']))&(~pd.isnull(df_tmp['seller_id']))]
        df_tmp['catalog_item_id'] = df_tmp['catalog_item_id'].astype('int')
        df_tmp['catalog_item_id'] = df_tmp['catalog_item_id'].astype('str')
        df_tmp['seller_id'] = df_tmp['seller_id'].astype('int')
        df_tmp['seller_id'] = df_tmp['seller_id'].astype('str')
        df_tmp.drop_duplicates(subset = ['catalog_item_id','seller_id'], inplace=True)
        df_feat_all = df_feat_all.merge(df_tmp, on=['catalog_item_id', 'seller_id'], how='left')
        del df_tmp
        
    # read bounce features
    files_to_read = [x for x in files if x[-14:-4] == date and x[:6]=='bounce' and 'item' in x]
    for file in files_to_read:
        print('merge bounce features')
        df_tmp = pd.read_csv(file, index_col=0)
        df_tmp = df_tmp[(~pd.isnull(df_tmp['catalog_item_id']))&(~pd.isnull(df_tmp['seller_id']))]
        df_tmp['catalog_item_id'] = df_tmp['catalog_item_id'].astype('int')
        df_tmp['catalog_item_id'] = df_tmp['catalog_item_id'].astype('str')
        df_tmp['seller_id'] = df_tmp['seller_id'].astype('int')
        df_tmp['seller_id'] = df_tmp['seller_id'].astype('str')
        df_tmp.drop_duplicates(subset = ['catalog_item_id','seller_id'], inplace=True)
        df_feat_all = df_feat_all.merge(df_tmp, on=['catalog_item_id', 'seller_id'], how='left')
        del df_tmp
        
    df_feat_all.to_csv('../sample_data_full_feat/df_feat_all_coop_item_'+date+'.csv')
    del df_feat_all

working on date 2023-01-16


/opt/conda/anaconda/lib/python3.6/site-packages/IPython/core/interactiveshell.py:3072: DtypeWarning: Columns (1) have mixed types. Specify dtype option on import or set low_memory=False.
  interactivity=interactivity, compiler=compiler, result=result)
/home/mds_backfill/.local/lib/python3.6/site-packages/numpy/lib/arraysetops.py:580: FutureWarning: elementwise comparison failed; returning scalar instead, but in the future will perform elementwise comparison
  mask |= (ar1 == a)


merge sem coop features
merge sem coop features
merge sem coop features
merge sem coop features
merge sem coop features
merge sem coop features
merge sem coop features
merge sem coop features
merge sem coop features
merge sem coop features
merge catalog features


/opt/conda/anaconda/lib/python3.6/site-packages/IPython/core/interactiveshell.py:3072: DtypeWarning: Columns (1,3,4,5,6,7,8,10) have mixed types. Specify dtype option on import or set low_memory=False.
  interactivity=interactivity, compiler=compiler, result=result)


merge site features
merge site features
merge bounce features
merge bounce features
working on date 2023-01-09
merge sem coop features
merge sem coop features
merge sem coop features
merge sem coop features
merge sem coop features
merge sem coop features
merge sem coop features
merge sem coop features
merge sem coop features
merge sem coop features
merge catalog features
merge site features
merge site features


In [16]:
for date in ref_dates[2:][::-1]:
    print('working on date '+date)
    # read header
    file = 'df_header_'+date+'.csv'
    df_feat_all = pd.read_csv(file, index_col=0)
    
    df_feat_all = df_feat_all[(~pd.isnull(df_feat_all['catalog_item_id']))&(~pd.isnull(df_feat_all['seller_id']))]
    df_feat_all['catalog_item_id'] = df_feat_all['catalog_item_id'].astype('int')
    df_feat_all['catalog_item_id'] = df_feat_all['catalog_item_id'].astype('str')
    df_feat_all['seller_id'] = df_feat_all['seller_id'].astype('int')
    df_feat_all['seller_id'] = df_feat_all['seller_id'].astype('str')
    df_feat_all.drop_duplicates(subset = 'adid', inplace=True)
    
    # drop off duplicates catalog_item_id, seller_id
    
#     # read targets
#     file = 'sem_l_'+date+'.csv'
#     df_tmp = pd.read_csv(file, index_col=0)
#     df_feat_all = df_tmp.merge(df_feat_all, left_on = 'catalog_item_id', right_on = 'catalog_item_id', how='left')
    
    # read sem features
    files_to_read = [x for x in files if x[-14:-4] == date and x[:4]=='sem_']
    
    # left join to ensure item coverage is based on target coverage
    for file in files_to_read:
        print('merge sem features')
        df_tmp = pd.read_csv(file, index_col=0)
        df_tmp = df_tmp[(~pd.isnull(df_tmp['catalog_item_id']))&(~pd.isnull(df_tmp['seller_id']))]
        df_tmp['catalog_item_id'] = df_tmp['catalog_item_id'].astype('int')
        df_tmp['catalog_item_id'] = df_tmp['catalog_item_id'].astype('str')
        df_tmp['seller_id'] = df_tmp['seller_id'].astype('int')
        df_tmp['seller_id'] = df_tmp['seller_id'].astype('str')
        df_tmp.drop_duplicates(subset = 'adid', inplace=True)
        df_feat_all = df_feat_all.merge(df_tmp, on=['catalog_item_id', 'seller_id', 'source_id', 'adid', 'is_mobile'], how='left')
        del df_tmp
        
    # read catalog feature
    files_to_read = [x for x in files if x[-14:-4] == date and x[:7]=='catalog']
    for file in files_to_read:
        print('merge catalog features')
        df_tmp = pd.read_csv(file, index_col=0)
        df_tmp = df_tmp[(~pd.isnull(df_tmp['catalog_item_id']))&(~pd.isnull(df_tmp['seller_id']))]
        df_tmp['catalog_item_id'] = df_tmp['catalog_item_id'].astype('int')
        df_tmp['catalog_item_id'] = df_tmp['catalog_item_id'].astype('str')
        df_tmp['seller_id'] = df_tmp['seller_id'].astype('int')
        df_tmp['seller_id'] = df_tmp['seller_id'].astype('str')
        df_tmp.drop_duplicates(subset = ['catalog_item_id','seller_id'], inplace=True)
        df_feat_all = df_feat_all.merge(df_tmp, on=['catalog_item_id', 'seller_id'], how='left')
        del df_tmp
        
    # read site features
    files_to_read = [x for x in files if x[-14:-4] == date and x[:4]=='site' and 'item' in x]
    for file in files_to_read:
        print('merge site features')
        df_tmp = pd.read_csv(file, index_col=0)
        df_tmp = df_tmp[(~pd.isnull(df_tmp['catalog_item_id']))&(~pd.isnull(df_tmp['seller_id']))]
        df_tmp['catalog_item_id'] = df_tmp['catalog_item_id'].astype('int')
        df_tmp['catalog_item_id'] = df_tmp['catalog_item_id'].astype('str')
        df_tmp['seller_id'] = df_tmp['seller_id'].astype('int')
        df_tmp['seller_id'] = df_tmp['seller_id'].astype('str')
        df_tmp.drop_duplicates(subset = ['catalog_item_id','seller_id'], inplace=True)
        df_feat_all = df_feat_all.merge(df_tmp, on=['catalog_item_id', 'seller_id'], how='left')
        del df_tmp
        
    # read bounce features
    files_to_read = [x for x in files if x[-14:-4] == date and x[:6]=='bounce' and 'item' in x]
    for file in files_to_read:
        print('merge bounce features')
        df_tmp = pd.read_csv(file, index_col=0)
        df_tmp = df_tmp[(~pd.isnull(df_tmp['catalog_item_id']))&(~pd.isnull(df_tmp['seller_id']))]
        df_tmp['catalog_item_id'] = df_tmp['catalog_item_id'].astype('int')
        df_tmp['catalog_item_id'] = df_tmp['catalog_item_id'].astype('str')
        df_tmp['seller_id'] = df_tmp['seller_id'].astype('int')
        df_tmp['seller_id'] = df_tmp['seller_id'].astype('str')
        df_tmp.drop_duplicates(subset = ['catalog_item_id','seller_id'], inplace=True)
        df_feat_all = df_feat_all.merge(df_tmp, on=['catalog_item_id', 'seller_id'], how='left')
        del df_tmp
        
    df_feat_all.to_csv('../sample_data_full_feat/df_feat_all_item_'+date+'.csv')
    del df_feat_all

working on date 2022-11-20
merge sem features
merge sem features
merge sem features
merge sem features
merge sem features
merge sem features
merge sem features
merge sem features
merge sem features
merge sem features
merge catalog features
merge site features
merge site features
merge bounce features
merge bounce features
working on date 2022-11-13
merge sem features
merge sem features
merge sem features
merge sem features
merge sem features
merge sem features
merge sem features
merge sem features
merge sem features
merge sem features
merge catalog features
merge site features
merge site features
merge bounce features
merge bounce features
working on date 2022-11-06
merge sem features
merge sem features
merge sem features
merge sem features
merge sem features
merge sem features
merge sem features
merge sem features
merge sem features
merge sem features
merge catalog features
merge site features
merge site features
merge bounce features
merge bounce features
working on date 2022-10-30
m